# MobileNetV2 road-follower training

Regresses `[distance, angle]` from a 224x224 camera frame, mirroring the
ResNet-18 pipeline in `train_model2.ipynb`. The best model (lowest test MSE)
is saved to both **`.pth`** (state-dict) and **`.onnx`** for deployment.


In [1]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import csv
import PIL.Image
import os
import numpy as np
import json
import math
import pandas as pd

## 1. Build labels.csv from the raw annotations

Same target encoding as `train_model2.ipynb`: `distance` is the normalized forward progress, `angle` is the (normalized, flipped) steering angle.

In [2]:
LABELS_JSON = "./dataset/raw/labeled/labels.json"
LABELS_CSV = "./dataset/raw/labeled/labels.csv"
IMGS_DIR = "./dataset/raw/imgs"
IMG_W, IMG_H = 224, 224

with open(LABELS_JSON, "r", encoding="utf-8") as f:
    labels = json.load(f)

labels = {k: v for k, v in labels.items() if v.get("x") is not None and v.get("y") is not None}

for key, item in labels.items():
    item["distance"] = item["y"]
    item["angle"] = math.atan2(item["x"] - IMG_W / 2, IMG_H - item["y"])

file_names = list(labels.keys())
distances = [1 - (item["distance"] / 224) for item in labels.values()]
angles = [item["angle"] for item in labels.values()]
max_abs_angle = max(abs(a) for a in angles)
angles = [-(angle / max_abs_angle) for angle in angles]

pd.DataFrame({"file_name": file_names, "distance": distances, "angle": angles}).to_csv(LABELS_CSV, index=False)
print(f"Wrote {len(file_names)} rows to {LABELS_CSV}")

Wrote 973 rows to ./dataset/raw/labeled/labels.csv


## 2. Dataset

Identical preprocessing to the existing pipeline so the saved weights are compatible with `show_results.py`: ColorJitter, resize 224, to_tensor, RGB->BGR channel flip, ImageNet normalization.

In [3]:
class XYDataset(torch.utils.data.Dataset):

    def __init__(self, csv_path, imgs_dir, random_hflips=False):
        self.imgs_dir = imgs_dir
        self.random_hflips = random_hflips
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)

        self.samples = []
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row) < 3:
                    continue
                img_path = os.path.join(imgs_dir, row[0])
                if os.path.exists(img_path):
                    self.samples.append((img_path, float(row[1]), float(row[2])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, distance, angle = self.samples[idx]
        image = PIL.Image.open(image_path)

        if self.random_hflips and np.random.rand() > 0.5:
            image = transforms.functional.hflip(image)
            angle = -angle

        image = self.color_jitter(image)
        image = transforms.functional.resize(image, (224, 224))
        image = transforms.functional.to_tensor(image)
        image = image.numpy()[:, :, ::-1].copy()
        angle = -angle
        image = torch.from_numpy(image)
        image = transforms.functional.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

        return image, torch.tensor([distance, angle]).float()

dataset = XYDataset(LABELS_CSV, IMGS_DIR, random_hflips=True)
print(f'Total samples: {len(dataset)}')

Total samples: 973


## 3. Train / test split and loaders

In [4]:
test_percent = 0.1
num_test = int(test_percent * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [len(dataset) - num_test, num_test])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=0)
print(f'Train: {len(train_dataset)}  Test: {len(test_dataset)}')

Train: 876  Test: 97


## 4. MobileNetV2 model

Pretrained backbone with the classifier replaced by a 2-output regression head.

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = torch.nn.Linear(model.last_channel, 2)
model = model.to(device)
print(f'Device: {device}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\konst/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 23.6MB/s]


Device: cuda
Trainable params: 2,226,434


## 5. Train, keeping the best checkpoint

In [7]:
from tqdm.auto import tqdm

NUM_EPOCHS = 15
BEST_MODEL_PATH = 'models/jetbot_cnn_mobilenet_v2.pth'
best_loss = 1e9

optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0
    train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]', leave=False)
    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.mse_loss(outputs, labels)
        train_loss += float(loss)
        loss.backward()
        optimizer.step()
        train_bar.set_postfix(loss=float(loss))
    train_loss /= len(train_loader)

    model.eval()
    test_loss = 0.0
    test_bar = tqdm(test_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [test]', leave=False)
    with torch.no_grad():
        for images, labels in test_bar:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = F.mse_loss(outputs, labels)
            test_loss += float(loss)
            test_bar.set_postfix(loss=float(loss))
    test_loss /= len(test_loader)

    improved = test_loss < best_loss
    if improved:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_loss = test_loss
    print(f'Epoch {epoch+1:>3}/{NUM_EPOCHS} | train_loss={train_loss:.6f} | '
          f'test_loss={test_loss:.6f} | best={best_loss:.6f}{" *" if improved else ""}')

print(f'\nBest test MSE: {best_loss:.6f}  ->  {BEST_MODEL_PATH}')

Epoch 1/15 [train]:   0%|          | 0/55 [00:00<?, ?it/s]

Epoch 1/15 [test]:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch   1/15 | train_loss=0.053770 | test_loss=0.038050 | best=0.038050 *


Epoch 2/15 [train]:   0%|          | 0/55 [00:00<?, ?it/s]

Epoch 2/15 [test]:   0%|          | 0/7 [00:00<?, ?it/s]

...

Epoch 14/15 [test]:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch  14/15 | train_loss=0.007691 | test_loss=0.013471 | best=0.011386


Epoch 15/15 [train]:   0%|          | 0/55 [00:00<?, ?it/s]

Epoch 15/15 [test]:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch  15/15 | train_loss=0.008113 | test_loss=0.042730 | best=0.011386

Best test MSE: 0.011386  ->  models/jetbot_cnn_mobilenet_v2.pth


## 6. Evaluate the best checkpoint

In [9]:
eval_model = models.mobilenet_v2(weights=None)
eval_model.classifier[1] = torch.nn.Linear(eval_model.last_channel, 2)
eval_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
eval_model = eval_model.to(device).eval()

squared_error_sum = torch.zeros(2, device=device)
n_samples = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = eval_model(images)
        squared_error_sum += ((outputs - labels) ** 2).sum(dim=0)
        n_samples += labels.size(0)

per_target_mse = (squared_error_sum / n_samples).cpu().numpy()
mse = float(per_target_mse.mean())
print(f'Samples evaluated: {n_samples}')
print(f'MSE (distance):    {per_target_mse[0]:.6f}')
print(f'MSE (angle):       {per_target_mse[1]:.6f}')
print(f'MSE (overall):     {mse:.6f}')

Samples evaluated: 97
MSE (distance):    0.003481
MSE (angle):       0.049002
MSE (overall):     0.026242


## 7. Export the best model to ONNX

Exports the best checkpoint with a fixed `1x3x224x224` input plus a dynamic batch axis, so it can run under ONNX Runtime / TensorRT on the JetBot.

In [10]:
ONNX_MODEL_PATH = 'models/jetbot_cnn_mobilenet_v2.onnx'
os.makedirs(os.path.dirname(ONNX_MODEL_PATH), exist_ok=True)

eval_model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)

# dynamo=False uses the legacy TorchScript exporter. On torch>=2.9 the new
# torch.export (dynamo) path is the default, but it needs the onnxscript
# package and prints unicode that crashes the Windows cp1250 console.
torch.onnx.export(
    eval_model,
    dummy_input,
    ONNX_MODEL_PATH,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    dynamo=False,
)
print(f'Exported ONNX model -> {ONNX_MODEL_PATH}')
# Validate the exported graph and check parity with PyTorch.
import onnx
onnx_model = onnx.load(ONNX_MODEL_PATH)
onnx.checker.check_model(onnx_model)
print('ONNX model is valid.')

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(ONNX_MODEL_PATH, providers=['CPUExecutionProvider'])
    onnx_out = sess.run(None, {'input': dummy_input.cpu().numpy()})[0]
    with torch.no_grad():
        torch_out = eval_model(dummy_input).cpu().numpy()
    max_diff = float(np.abs(onnx_out - torch_out).max())
    print(f'Max |torch - onnx| on dummy input: {max_diff:.2e}')
except ImportError:
    print('onnxruntime not installed - skipping parity check.')

Exported ONNX model -> models/jetbot_cnn_mobilenet_v2.onnx
ONNX model is valid.
Max |torch - onnx| on dummy input: 3.14e-04
